# Чекпоинт 7

Проект: «Идентификация искусственно сгенерированных текстов: разработка методов для обеспечения информационной достоверности».


## Краткая цель

Базовая цель — воспроизвести PAWN и проверить, дают ли дополнительные признаки от второй LLM прирост качества.

Основные направления доработок: вторая frozen LLM, cross-model метрики, метрики второй модели, fusion hidden states, sequence-level aggregate metrics.


## Импорты 


In [21]:
from pathlib import Path
import json
import polars as pl

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)

ROOT = Path.cwd()


## 0.1 Датасет

В качестве датасета для всех экспериментов используется [MAGE](https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18):

- Paper: https://arxiv.org/pdf/2305.13242

- GitHub: https://github.com/yafuly/MAGE?tab=readme-ov-file#-dataset

- Hugging Face: https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18


Датасет содержит большое количество доменов и моделей, что позволяет полноценно оценить работу детектора.

Более того, этот датасет используется в оригинальной статье [PAWN](https://www.sciencedirect.com/science/article/pii/S156625352500538X?ref=pdf_download&fr=RR-2&rr=9f97f735c8398b88)

Изначально в датасете использовалось следующее распределение по категориям: 

1 - Human-written, 0 - Machine-generated

Чтобы соответствовать оригинальной работе, категории были поменяны местами: 

0 - Human-written, 1 - Machine-generated


In [22]:
df_mage_train = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/train.csv")
df_mage_valid = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/valid.csv")
df_mage_test = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv")


In [23]:
print(f"Кол-во наблюдений в трейне: {df_mage_train.height}")
print(f"Кол-во наблюдений в валидации: {df_mage_valid.height}")
print(f"Кол-во наблюдений в тесет: {df_mage_test.height}")


Кол-во наблюдений в трейне: 319071
Кол-во наблюдений в валидации: 56792
Кол-во наблюдений в тесет: 60743


In [24]:
# Соотношение классов в трейне
df_mage_train.group_by("label").agg(pl.len().alias("count"))


label,count
i64,u32
0,93318
1,225753


In [25]:
# Соотношение классов в валидации
df_mage_valid.group_by("label").agg(pl.len().alias("count"))


label,count
i64,u32
1,27993
0,28799


In [26]:
# Соотношение классов в тесте
df_mage_test.group_by("label").agg(pl.len().alias("count"))


label,count
i64,u32
1,30265
0,30478


### Важное замечание

Для проведения экспериментов использовалась сбалансированная подвыборка из трейна и валидации на 5000 и 2000 наблюдений, соответственно. Тестовая выборка использовалась полностью для замера финальных метрик.


In [27]:
df_mage_train_sampled = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled_inv.csv")
df_mage_valid_sampled = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled_inv.csv")


In [28]:
# Соотношение классов в трейне
df_mage_train_sampled.group_by("label").agg(pl.len().alias("count"))


label,count
i64,u32
1,2500
0,2500


In [29]:
# Соотношение классов в валидации
df_mage_valid_sampled.group_by("label").agg(pl.len().alias("count"))


label,count
i64,u32
0,1000
1,1000


## 0.2 Модели

Для PAWN-экспериментов использовалась модель ```meta-llama/Llama-3.2-1B-Instruct``` и ее базовая версия — ```meta-llama/Llama-3.2-1B```.

Для отдельного encoder-baseline использовалась ```FacebookAI/roberta-base```. Эти эксперименты добавлены в таблицы как BERT/RoBERTa baseline для сравнения с PAWN-вариантами.


## Загрузка результатов


In [30]:
RUNS = {
    "Llama Instruct baseline": {
        "group": "PAWN baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_instuct/test_metrics.json",
    },
    "Llama Base baseline": {
        "group": "PAWN baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_base/test_metrics.json",
    },
    "Llama Instruct full baseline": {
        "group": "PAWN baseline",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_instruct_full/test_metrics.json",
    },
    "Llama Base full baseline": {
        "group": "PAWN baseline",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_base_full/test_metrics.json",
    },
    "RoBERTa sampled, best by AUROC": {
        "group": "BERT baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/bert/mage_sampled_roberta_auc/test_metrics.json",
    },
    "RoBERTa sampled, best by recall": {
        "group": "BERT baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/bert/mage_sampled_roberta_recall/test_metrics.json",
    },
    "RoBERTa full, best by AUROC": {
        "group": "BERT baseline",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/bert/mage_full_roberta_auc/test_metrics.json",
    },
    "RoBERTa full, best by recall": {
        "group": "BERT baseline",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/bert/mage_full_roberta_recall/test_metrics.json",
    },
    "Llama Base + agg metrics": {
        "group": "Aggregate metrics",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_base_agg_metrics/test_metrics.json",
    },
    "Llama Instruct + agg metrics": {
        "group": "Aggregate metrics",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_instruct_agg_metrics/test_metrics.json",
    },
    "Llama Base + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_base_hs_uniform/test_metrics.json",
    },
    "Llama Instruct + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_instruct_hs_uniform/test_metrics.json",
    },
    "Llama Instruct + agg + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/single_model/mage_llama_instruct_agg_metrics_hs_uniform/test_metrics.json",
    },
    "Two models + XPPL": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_xppl/test_metrics.json",
    },
    "Two models + second metrics + XPPL": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl/test_metrics.json",
    },
    "Two models + second HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_hs/test_metrics.json",
    },
    "Two models + metrics + XPPL + HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs/test_metrics.json",
    },
    "Two models + metrics + XPPL + uniform HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform/test_metrics.json",
    },
    "Two models + metrics + XPPL + uniform HS full": {
        "group": "Two models",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_full/test_metrics.json",
    },
    "PAWN++ sampled": {
        "group": "PAWN++",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics/test_metrics.json",
    },
    "PAWN++ full": {
        "group": "PAWN++",
        "split": "full",
        "path": "pawn++/experiments/MAGE/cross_domains_cross_models/runs/pawn/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics_full/test_metrics.json",
    },
    "OOD RoBERTa full AUROC on GPT": {
        "group": "OOD BERT baseline",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_full_roberta_auc_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD RoBERTa full AUROC on GPT para": {
        "group": "OOD BERT baseline",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_full_roberta_auc_ood_eval/test_ood_gpt_para_metrics.json",
    },
    "OOD RoBERTa full recall on GPT": {
        "group": "OOD BERT baseline",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_full_roberta_recall_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD RoBERTa full recall on GPT para": {
        "group": "OOD BERT baseline",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_full_roberta_recall_ood_eval/test_ood_gpt_para_metrics.json",
    },
    "OOD Llama Base full on GPT": {
        "group": "OOD PAWN baseline",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_base_full_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD Llama Base full on GPT para": {
        "group": "OOD PAWN baseline",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_base_full_ood_eval/test_ood_gpt_para_metrics.json",
    },
    "OOD Llama Instruct full on GPT": {
        "group": "OOD PAWN baseline",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_full_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD Llama Instruct full on GPT para": {
        "group": "OOD PAWN baseline",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_full_ood_eval/test_ood_gpt_para_metrics.json",
    },
    "OOD Two models full on GPT": {
        "group": "OOD PAWN++",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_full_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD Two models full on GPT para": {
        "group": "OOD PAWN++",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_full_ood_eval/test_ood_gpt_para_metrics.json",
    },
    "OOD PAWN++ full on GPT": {
        "group": "OOD PAWN++",
        "split": "ood_gpt",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics_full_ood_eval/test_ood_gpt_metrics.json",
    },
    "OOD PAWN++ full on GPT para": {
        "group": "OOD PAWN++",
        "split": "ood_gpt_para",
        "path": "pawn++/experiments/MAGE/ood/runs/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics_full_ood_eval/test_ood_gpt_para_metrics.json",
    },
}


In [31]:
def metric_value(metrics: dict, name: str) -> float:
    test_key = f"test_{name}"
    if test_key in metrics:
        return metrics[test_key]
    return metrics[name]


def load_results(experiments: list[str]) -> pl.DataFrame:
    rows = []
    for experiment in experiments:
        run = RUNS[experiment]
        metrics = json.loads((ROOT / run["path"]).read_text())

        human_recall = metric_value(metrics, "human_recall")
        ai_recall = metric_value(metrics, "ai_recall")
        avg_recall = metrics.get("test_avg_recall", metrics.get("avg_recall", (human_recall + ai_recall) / 2))

        rows.append({
            "group": run["group"],
            "experiment": experiment,
            "split": run["split"],
            "AUCROC": metric_value(metrics, "roc_auc"),
            "accuracy": metric_value(metrics, "accuracy"),
            "human_recall": human_recall,
            "ai_recall": ai_recall,
            "avg_recall": avg_recall,
            "f1_macro": metric_value(metrics, "f1_macro"),
        })

    return (
        pl.DataFrame(rows)
        .with_columns(pl.col("AUCROC", "accuracy", "human_recall", "ai_recall", "avg_recall", "f1_macro").round(4))
    )


## Оригинальная PAWN-архитектура

Оригинальный PAWN берет logits и hidden states одной замороженной LLM. Из logits считаются token-level метрики, hidden states используются gate-сетью для взвешивания токенов, после чего агрегированный вектор передается в Aggregate NN.


![Original PAWN](pawn++/pawn_images/pawn_original.jpg)


### Бейзлайны

Базовая модель PAWN без доработок на подвыборке из 5000 тысяч наблюдений и на полном датасете MAGE (full)


In [32]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "RoBERTa sampled, best by AUROC",
    "RoBERTa sampled, best by recall",
    "Llama Instruct full baseline",
    "Llama Base full baseline",
    "RoBERTa full, best by AUROC",
    "RoBERTa full, best by recall",
]).sort("AUCROC", descending=True)


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""BERT baseline""","""RoBERTa full, best by recall""","""full""",0.9801,0.9066,0.8506,0.9623,0.9064,0.9063
"""PAWN baseline""","""Llama Base full baseline""","""full""",0.9782,0.9331,0.9447,0.9215,0.9331,0.9331
"""BERT baseline""","""RoBERTa full, best by AUROC""","""full""",0.9773,0.8829,0.7909,0.9741,0.8825,0.8818
"""PAWN baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9286,0.9285
"""BERT baseline""","""RoBERTa sampled, best by recall""","""sampled""",0.9335,0.8534,0.8179,0.8886,0.8533,0.8532
"""BERT baseline""","""RoBERTa sampled, best by AUROC""","""sampled""",0.9213,0.7223,0.4747,0.9681,0.7214,0.7039
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039
"""PAWN baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.8001,0.7996


## 1. Вторая frozen LLM

Основная идея всего расширения оригинального PAWN — добавить вторую замороженную модель. Она может отдавать свои logits и hidden states, на основе которых можно считать дополнительные признаки для улучшения ихсодной архитектуры PAWN.


![Second Frozen Model](pawn++/pawn_images/1_second_frozen_model.png)


## 2. Cross-model metrics (XPPL)

Был добавлен признак XPPL в стиле [Binoculars](https://arxiv.org/abs/2401.12070): одна модель дает log-probabilities, другая — probability distribution.


![Cross Metrics XPPL](pawn++/pawn_images/2_cross_metrics_xppl.png)


### Результаты XPPL


In [33]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Two models + XPPL",
    "Two models + second metrics + XPPL",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""PAWN baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.8001,0.7996
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039
"""Two models""","""Two models + XPPL""","""sampled""",0.8676,0.7848,0.7235,0.8457,0.7846,0.7839
"""Two models""","""Two models + second metrics + XPPL""","""sampled""",0.9092,0.8259,0.78,0.8713,0.8257,0.8254


Комментарий: XPPL отдельно оказался слабым, но результат заметно улучшился при добавлении token-level метрик второй модели. Это значит, что один cross-score недостаточно информативен, но в связке с полным набором признаков второй модели он становится полезнее.


## 3. Метрики второй модели

В базовой реализации используются метрики, рассчитанные по logits основной модели: `max_log_probs`, `entropy`, `next_token_log_probs`, `rank`, `top_p`.

Для второй frozen LLM были добавлены собственные token-level метрики. Они конкатенируются с метриками основной модели и проходят через Metrics NN.


![Second Model Metrics](pawn++/pawn_images/3_second_model_metrics.png)


### Результаты second-model metrics


In [34]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Two models + XPPL",
    "Two models + second metrics + XPPL",
    "Two models + metrics + XPPL + HS",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""PAWN baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.8001,0.7996
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039
"""Two models""","""Two models + XPPL""","""sampled""",0.8676,0.7848,0.7235,0.8457,0.7846,0.7839
"""Two models""","""Two models + second metrics + XPPL""","""sampled""",0.9092,0.8259,0.78,0.8713,0.8257,0.8254
"""Two models""","""Two models + metrics + XPPL + HS""","""sampled""",0.9045,0.8318,0.8348,0.8288,0.8318,0.8318


Комментарий: это одно из наиболее полезных расширений. Метрики второй модели добавляют информацию о том, как другая LLM оценивает те же токены, и это дает прирост относительно использования только XPPL.


## 4. Hidden states fusion

В оригинальном PAWN используется последний hidden state. 

В улучшенной версии были проверены варианты использования информации сразу из всех hidden-state. Это реализовано за счет усреднения предварительно отнормализованных hidden states модели по каждому токену. Это делается параллельно для двух моделей, после чего их hidden states конкатенируются и подаются в Weights NN вместе в вектором, кодирующим позицию.


Для модели $m$ и токена $t$ усредненный hidden-state считается так:

$$
\tilde{h}^{(m)}_t =
\frac{1}{K}
\sum_{k=1}^{K}
\operatorname{LayerNorm}\left(h^{(m,k)}_t\right),
$$

где $h^{(m,k)}_t$ — hidden state токена $t$ на слое $k$, $K$ — число используемых слоев модели, а $\tilde{h}^{(m)}_t$ — итоговый усредненый hidden state.

Для двух моделей вход в Weights NN формируется как:

$$
g_t =
\left[
\tilde{h}^{(1)}_t ;
\tilde{h}^{(1)}_{t+1} ;
\tilde{h}^{(2)}_t ; 
\tilde{h}^{(2)}_{t+1} ;
p_t
\right]
$$

где $p_t$ — вектор с позиционным кодированием токена.


![Hidden States Fusion](pawn++/pawn_images/4_hidden_states_fusion.png)


### Результаты hidden-state fusion


In [35]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Llama Base + uniform HS",
    "Llama Instruct + uniform HS",
    "Two models + metrics + XPPL + uniform HS",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""PAWN baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.8001,0.7996
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039
"""Hidden-state fusion""","""Llama Base + uniform HS""","""sampled""",0.9273,0.8469,0.8368,0.857,0.8469,0.8469
"""Hidden-state fusion""","""Llama Instruct + uniform HS""","""sampled""",0.9096,0.8244,0.794,0.8545,0.8243,0.8242
"""Two models""","""Two models + metrics + XPPL + uniform HS""","""sampled""",0.9288,0.8557,0.8503,0.8609,0.8556,0.8556


Комментарий: uniform fusion оказался самым стабильным улучшением. Особенно заметен результат `Llama Base + uniform HS`, который стал лучшим single-model вариантом на сэмплированной выборке.


## 5. Aggregated sequence-level metrics

Были добавлены sequence-level признаки: статистики по log-likelihood / surprisal всей последовательности и их производным. 

Исходя из работы [DivEye](https://arxiv.org/pdf/2509.18880), добавление агрегированных метрик на основе log-likelihood токенов может значительно улучшать качество детекторов.

Проверялись разные способы добавления агрегированных метрик в конец модели PAWN: 
- Конкатенация агрегированных метрик с финальным скором PAWN 
- Конкатенация агрегированных метрик с финальным вектором признаков перед подачей в `aggregate_nn`
- [FiLM](https://arxiv.org/pdf/1709.07871)-like добавление агрегированных метрик к финальному вектору признаков перед подачей в `aggregate_nn`

По итогам экспериментов лучший результат показал последний вариант. 

Математически пусть $F$ — финальный вектор признаков PAWN перед подачей в Aggregate NN.

Агрегированные sequence-level метрики обозначим как $s$. Сначала они нормализуются:

$$
\hat{s} = \operatorname{LayerNorm}(s).
$$

Затем из них предсказываются FiLM-параметры $\gamma$ и $\beta$:

$$
[\gamma, \beta] = W_{\text{film}} \hat{s} + b_{\text{film}},
$$

где $\gamma$ и $\beta$ имеют ту же размерность, что и $F$.

FiLM-like преобразование вектора $F$ считается как:

$$
F_{\text{film}} =
(1 + \gamma) \odot F + \beta.
$$

После этого уже модифицированный вектор передается в Aggregate NN:

$$
\text{logit} =
\operatorname{AggregateNN}(F_{\text{film}}).
$$

$W_{\text{film}}$ и  $b_{\text{film}}$ инициализируются нулями:

$$
W_{\text{film}} = 0, \quad b_{\text{film}} = 0.
$$

Тогда в начале обучения:

$$
\gamma = 0, \quad \beta = 0,
$$

и поэтому:

$$
F_{\text{film}} = F
$$


![Aggregated Metrics](pawn++/pawn_images/5_aggregated_metrics.png)


![Aggregate Metrics Fusion](pawn++/pawn_images/6_aggregated_metrics_fusion_film.png)


### Результаты


In [36]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Llama Instruct + agg metrics",
    "Llama Base + agg metrics",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""PAWN baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.8001,0.7996
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039
"""Aggregate metrics""","""Llama Instruct + agg metrics""","""sampled""",0.8821,0.8055,0.7766,0.8341,0.8054,0.8053
"""Aggregate metrics""","""Llama Base + agg metrics""","""sampled""",0.8738,0.7862,0.7609,0.8113,0.7861,0.786


Комментарий: агрегированные метрики не дали стабильного прироста в single-model setup. Качество для instruct-tuned модели выросло, однако для базовой модели, наоборот, упало. Вероятно, часть информации уже извлекается PAWN через token-level признаки, поэтому sequence-level статистики не дают значимого прироста по качеству.


## Финальная архитектура PAWN++

PAWN++ объединяет основные расширения: вторую frozen LLM, second-model token metrics, cross-model metrics / XPPL, hidden-state fusion и aggregate metrics.


![PAWN++](pawn++/pawn_images/pawn++.png)


### Итоговые результаты PAWN++


In [37]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "RoBERTa sampled, best by AUROC",
    "RoBERTa sampled, best by recall",
    "PAWN++ sampled",
    "Llama Instruct full baseline",
    "Llama Base full baseline",
    "RoBERTa full, best by AUROC",
    "RoBERTa full, best by recall",
    "PAWN++ full",
]).sort("AUCROC", descending=True)


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""PAWN++""","""PAWN++ full""","""full""",0.9836,0.9515,0.972,0.9311,0.9516,0.9515
"""BERT baseline""","""RoBERTa full, best by recall""","""full""",0.9801,0.9066,0.8506,0.9623,0.9064,0.9063
"""PAWN baseline""","""Llama Base full baseline""","""full""",0.9782,0.9331,0.9447,0.9215,0.9331,0.9331
"""BERT baseline""","""RoBERTa full, best by AUROC""","""full""",0.9773,0.8829,0.7909,0.9741,0.8825,0.8818
"""PAWN baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9286,0.9285
"""BERT baseline""","""RoBERTa sampled, best by recall""","""sampled""",0.9335,0.8534,0.8179,0.8886,0.8533,0.8532
"""PAWN++""","""PAWN++ sampled""","""sampled""",0.9309,0.848,0.805,0.8908,0.8479,0.8477
"""BERT baseline""","""RoBERTa sampled, best by AUROC""","""sampled""",0.9213,0.7223,0.4747,0.9681,0.7214,0.7039
"""PAWN baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.804,0.8039


Комментарий: лучший результат получился у полной PAWN++ конфигурации на полном датасете. На сэмплированной выборке PAWN++ также оказался лучшим вариантом среди всех экспериментов, превосходя бейзлайны на более чем 3.5 процентных пункта.


## OOD-оценка на MAGE

Отдельно были проверены full-модели на OOD testbeds из MAGE: `test_ood_gpt` и `test_ood_gpt_para`. Эти результаты показывают, как модели переносятся на GPT-4 (OOD модель) и paraphrasing-attack сценарии.


In [38]:
load_results([
    "OOD RoBERTa full AUROC on GPT",
    "OOD RoBERTa full recall on GPT",
    "OOD Llama Base full on GPT",
    "OOD Llama Instruct full on GPT",
    "OOD Two models full on GPT",
    "OOD PAWN++ full on GPT",
    "OOD RoBERTa full AUROC on GPT para",
    "OOD RoBERTa full recall on GPT para",
    "OOD Llama Base full on GPT para",
    "OOD Llama Instruct full on GPT para",
    "OOD Two models full on GPT para",
    "OOD PAWN++ full on GPT para",
]).sort(["split", "AUCROC"], descending=[False, True])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""OOD PAWN++""","""OOD PAWN++ full on GPT""","""ood_gpt""",0.9842,0.9424,0.9304,0.9538,0.9421,0.9423
"""OOD PAWN++""","""OOD Two models full on GPT""","""ood_gpt""",0.9764,0.9142,0.8714,0.955,0.9132,0.9139
"""OOD PAWN baseline""","""OOD Llama Base full on GPT""","""ood_gpt""",0.9728,0.9033,0.8661,0.9388,0.9024,0.903
"""OOD PAWN baseline""","""OOD Llama Instruct full on GPT""","""ood_gpt""",0.9684,0.9123,0.8714,0.9512,0.9113,0.912
"""OOD BERT baseline""","""OOD RoBERTa full AUROC on GPT""","""ood_gpt""",0.9558,0.7522,0.4974,0.995,0.7462,0.7332
"""OOD BERT baseline""","""OOD RoBERTa full recall on GPT""","""ood_gpt""",0.9435,0.7964,0.601,0.9825,0.7918,0.787
"""OOD PAWN baseline""","""OOD Llama Base full on GPT para""","""ood_gpt_para""",0.7977,0.6473,0.8661,0.5431,0.7046,0.6445
"""OOD PAWN++""","""OOD PAWN++ full on GPT para""","""ood_gpt_para""",0.7913,0.6135,0.9304,0.4625,0.6965,0.6134
"""OOD PAWN++""","""OOD Two models full on GPT para""","""ood_gpt_para""",0.7733,0.619,0.8714,0.4988,0.6851,0.6177


## Финальная таблица

Все результаты собраны в одну таблицу и отсортированы по ROC-AUC.


In [39]:
results = load_results(list(RUNS.keys()))
results = results.sort("AUCROC", descending=True)
results


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,avg_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64,f64
"""OOD PAWN++""","""OOD PAWN++ full on GPT""","""ood_gpt""",0.9842,0.9424,0.9304,0.9538,0.9421,0.9423
"""Two models""","""Two models + metrics + XPPL + uniform HS full""","""full""",0.9841,0.9503,0.9635,0.9371,0.9503,0.9503
"""PAWN++""","""PAWN++ full""","""full""",0.9836,0.9515,0.972,0.9311,0.9516,0.9515
"""BERT baseline""","""RoBERTa full, best by recall""","""full""",0.9801,0.9066,0.8506,0.9623,0.9064,0.9063
"""PAWN baseline""","""Llama Base full baseline""","""full""",0.9782,0.9331,0.9447,0.9215,0.9331,0.9331
"""BERT baseline""","""RoBERTa full, best by AUROC""","""full""",0.9773,0.8829,0.7909,0.9741,0.8825,0.8818
"""OOD PAWN++""","""OOD Two models full on GPT""","""ood_gpt""",0.9764,0.9142,0.8714,0.955,0.9132,0.9139
"""PAWN baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9286,0.9285
"""OOD PAWN baseline""","""OOD Llama Base full on GPT""","""ood_gpt""",0.9728,0.9033,0.8661,0.9388,0.9024,0.903


## Выводы

1. Базовый PAWN воспроизведен и используется как бейзлайн.
2. Самое стабильное single-model улучшение — усреднение hidden-states.
3. Эксперименты показали, что token-level метрики и cross-model признаки второй модели дают значительный прирост в качестве детекции.
4. Агрегированные метрики дают прирост не во всех экспериментах, однако в используются финальной архитектуре PAWN++ и позволяют получить более высокие метрики качества.
5. Модель PAWN++, обученная на полном MAGE датасете, превосходит базовую модель PAWN по всем ключевым метрикам.
